In [58]:
import opensmile
import audiofile
import numpy as np
import pandas as pd

#FPS
VIDEO_FPS = 29.97

#ファイルの読み込み
ref_file_path = r"C:\Users\robotics\proj\Research\processed_annotation_data\ID30_annotation_processed.csv"
df = pd.read_csv(ref_file_path, header=None)

#CSVファイルのanswer部分を抽出
condition=df[0].str.contains('answer') 
answer_df = df[condition].copy()

#answerの番号でsort
answer_df['sort_key'] = answer_df[5].str.extract(r'(\d+)').astype(int)
sort_df = answer_df.sort_values(by='sort_key')
sort_df = sort_df.drop(columns=['sort_key'])

#start, endのフレームを抽出
start_list, end_list = sort_df[6], sort_df[7]
#print(start_list, end_list)

signal, sampling_rate = audiofile.read(r"D:\Douga_Niho\voice_data\ID30_audio.wav")

#opensmileの処理
result_list = []
for i in range(len(start_list)):
    label = sort_df[5]
    

    smile = opensmile.Smile(feature_set= r'C:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\config\egemaps\v02_niho\eGeMAPSv02.conf',
                            feature_level=opensmile.FeatureLevel.LowLevelDescriptors,)
    
    ID30_df = smile.process_file(
        r"D:\Douga_Niho\voice_data\ID30_audio.wav",
        start=(start_list.iloc[i]/VIDEO_FPS)*sampling_rate, #開始
        end=(end_list.iloc[i]/VIDEO_FPS)*sampling_rate #終了
        )
    ID30_df.insert(0, 'label', label)
    result_list.append(ID30_df)

ID30 = pd.concat(result_list, ignore_index=True)


ID30.to_csv(r"C:\Users\robotics\proj\Research\voice_csv_v2\ID30_voice.csv")
print(ID30)




c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
c:\Users\r

  label  Loudness_sma3  alphaRatio_sma3  hammarbergIndex_sma3  \
0   NaN            NaN              NaN                   NaN   
1   NaN            NaN              NaN                   NaN   
2   NaN            NaN              NaN                   NaN   
3   NaN            NaN              NaN                   NaN   
4   NaN            NaN              NaN                   NaN   
5   NaN            NaN              NaN                   NaN   
6   NaN            NaN              NaN                   NaN   
7   NaN            NaN              NaN                   NaN   
8   NaN            NaN              NaN                   NaN   
9   NaN            NaN              NaN                   NaN   

   slope0-500_sma3  slope500-1500_sma3  spectralFlux_sma3  mfcc1_sma3  \
0              NaN                 NaN                NaN         NaN   
1              NaN                 NaN                NaN         NaN   
2              NaN                 NaN                NaN        

c:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))


In [18]:
import opensmile
import audiofile
import numpy as np
import pandas as pd
from pathlib import Path

video_FPS = 29.97
annotation_path = r"C:\Users\robotics\proj\Research\processed_annotation_data\ID30_annotation_processed.csv"
wav_path = r"D:\Douga_Niho\voice_data\ID30_audio.wav"

#音声ファイルの読み込み
try:
    signal, sampling_rate = audiofile.read(wav_path)
except FileExistsError:
    print(f"ファイルが見つかりません。")
    exit()
if signal.ndim == 2: #1次元に
    signal = signal[0, :]

#音声区間の抽出
df = pd.read_csv(annotation_path, header=None)

answer_df = df[df[5].astype(str).str.startswith('answer')]
#print(answer_df)

#answerの番号でsort
answer_df['sort_key'] = answer_df[5].str.extract(r'(\d+)').astype(int)
sort_df = answer_df.sort_values(by='sort_key')
sort_df = sort_df.drop(columns=['sort_key'])
#print(sort_df)

print(f"音声ファイル'{wav_path}'の分析")
all_result = []

for index, row_df in sort_df.iterrows():
    #label, frameの抽出
    label = row_df[5]
    start_frame = row_df[6]
    end_frame = start_frame + 30
    
    #frame → second → sample
    start_sec = start_frame / video_FPS
    end_sec = end_frame / video_FPS
    start_sample = int(start_sec * sampling_rate)
    end_sample = int(end_sec * sampling_rate)
    signal_slice = signal[start_sample:end_sample] #使用する音声区間

    print(signal_slice)

    if signal_slice.size > 0:
        smile = opensmile.Smile(
            feature_set= r'C:\Users\robotics\proj\delirium\Lib\site-packages\opensmile\core\config\egemaps\v02_niho\eGeMAPSv02.conf',
            feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
        )

        result_df = smile.process_signal(signal_slice, sampling_rate)

        #時間情報とlabel情報を追加
        start_timedelta = pd.to_timedelta(start_sec, unit='s')
        result_df['original_start'] = result_df.index.get_level_values('start') + start_timedelta
        result_df['original_end'] = result_df.index.get_level_values('end') + start_timedelta
        result_df.insert(0, 'label', label)

        #結果を追加
        all_result.append(result_df)

    else:
        print("この区間はスキップ")

if all_result:
    print("\n結果を統合")
    final_df = pd.concat(all_result)

    #結果をCSVファイルとしてフォルダに保存
    output_folder = Path('./voice_csv_v2')
    file_path = output_folder / 'ID30_20ms.csv'
    final_df.to_csv(file_path, index=False)

    print(f"ファイルは {file_path} に保存されました。")

else:
    print("\n分析できる区間がない")

print("\nすべての処理が完了しました")    


C:\Users\robotics\AppData\Local\Temp\ipykernel_21992\2252180062.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  answer_df['sort_key'] = answer_df[5].str.extract(r'(\d+)').astype(int)


音声ファイル'D:\Douga_Niho\voice_data\ID30_audio.wav'の分析
[-0.0093689  -0.00942993 -0.00933838 ... -0.01074219 -0.01028442
 -0.01055908]
[-0.00860596 -0.0090332  -0.00946045 ... -0.00735474 -0.00753784
 -0.00784302]
[-0.00057983  0.0010376   0.00186157 ... -0.00241089 -0.00033569
  0.00067139]
[ 0.00723267  0.00765991  0.00793457 ... -0.01199341 -0.01159668
 -0.01104736]
[-0.0112915  -0.01330566 -0.01419067 ... -0.00769043 -0.00750732
 -0.0078125 ]
[ 0.00402832  0.00408936  0.00161743 ... -0.0072937  -0.00735474
 -0.00814819]
[-0.01269531 -0.01351929 -0.01513672 ... -0.01831055 -0.02087402
 -0.01989746]
[-0.00906372 -0.00872803 -0.00827026 ... -0.00946045 -0.00942993
 -0.00939941]
[-0.01092529 -0.01098633 -0.01107788 ... -0.01663208 -0.01599121
 -0.01541138]
[-0.00894165 -0.00918579 -0.01037598 ... -0.00921631 -0.0098877
 -0.01080322]

結果を統合


OSError: Cannot save file into a non-existent directory: 'voice_csv_v2'